# Long-Run CPML Stability

Long 2D and 3D simulations exercise forward propagation, reverse-mode CPML,
float32 saved wavefields, and both material gradients for orders 2, 4, and 8.
Initial fields and CPML memories are fixed in this FWI test, so validation
covers the returned fields and the material gradients requested by autograd.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


ModuleNotFoundError: No module named 'torch'

In [ ]:
import time
import torch

torch.manual_seed(2026)
CPU = torch.device("cpu")
CUDA = vu.selected_cuda_device()
DEVICES = [CPU] + ([] if CUDA is None else [CUDA])
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, CPU)
DEVICE_METADATA = [vu.runtime_metadata(DeepGPR, device) for device in DEVICES]
stress_rows = []


In [ ]:
def stress_case(device, order, dimension):
    if dimension == 2:
        shape, nt, pml, mode = (24, 32), 2000, 5, 2
        source_location = torch.tensor([[[12, 8, 0]]], dtype=torch.int32, device=device)
        receiver_location = torch.tensor(
            [[[10, 9, 0], [14, 9, 0]]], dtype=torch.int32, device=device
        )
        axis_x = torch.arange(shape[0], dtype=torch.float32, device=device)[:, None]
        axis_y = torch.arange(shape[1], dtype=torch.float32, device=device)[None, :]
        anomaly = torch.exp(
            -0.5 * (((axis_x - 14.0) / 3.0) ** 2 + ((axis_y - 18.0) / 4.0) ** 2)
        )
    else:
        shape, nt, pml, mode = (20, 24, 24), 1500, 5, 3
        source_location = torch.tensor([[[10, 7, 12]]], dtype=torch.int32, device=device)
        receiver_location = torch.tensor(
            [[[8, 7, 10], [12, 7, 14]]], dtype=torch.int32, device=device
        )
        x = torch.arange(shape[0], dtype=torch.float32, device=device)[:, None, None]
        y = torch.arange(shape[1], dtype=torch.float32, device=device)[None, :, None]
        z = torch.arange(shape[2], dtype=torch.float32, device=device)[None, None, :]
        anomaly = torch.exp(
            -0.5
            * (
                ((x - 11.0) / 3.0) ** 2
                + ((y - 14.0) / 4.0) ** 2
                + ((z - 13.0) / 3.5) ** 2
            )
        )
    er = (4.0 + 4.0 * anomaly).requires_grad_(True)
    se = (1.0e-3 + 1.9e-2 * anomaly).requires_grad_(True)
    source = DeepGPR.wavelet.ricker(5.0e8, nt, 2.0e-11, 2.0e-9).reshape(1, nt, 1).to(device)
    start = time.perf_counter()
    result = DeepGPR.compute(
        device=device,
        dx=0.02,
        dt=2.0e-11,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        pmlthick=pml,
        fdtd_order=order,
        mode=mode,
        model_gradient_sampling_interval=10,
        wavefield_storage_dtype=torch.float32,
        debug=True,
    )
    loss = result[-1].square().mean()
    loss.backward()
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start

    fields = (*result[1], *result[2], *result[3], result[-1])
    vu.assert_finite("long-run fields", *fields)
    vu.assert_finite("long-run gradients", er.grad, se.grad)
    boundary = vu.pml_boundary_mask(shape, pml, device)
    return {
        "device": str(device),
        "dimension": dimension,
        "order": order,
        "nt": nt,
        "elapsed_seconds": elapsed,
        "receiver_absmax": float(result[-1].detach().abs().max().cpu()),
        "er_gradient_absmax": float(er.grad.detach().abs().max().cpu()),
        "se_gradient_absmax": float(se.grad.detach().abs().max().cpu()),
        "er_boundary_absmax": vu.boundary_absmax(er.grad, boundary),
        "se_boundary_absmax": vu.boundary_absmax(se.grad, boundary),
    }


In [ ]:
for device in DEVICES:
    for dimension in (2, 3):
        for order in (2, 4, 8):
            row = stress_case(device, order, dimension)
            stress_rows.append(row)
            vu.record_check(
                CHECKS,
                f"long-run {dimension}D order {order} on {device}",
                row["receiver_absmax"] > 0.0
                and row["er_gradient_absmax"] > 0.0
                and row["se_gradient_absmax"] > 0.0
                and row["er_boundary_absmax"] == 0.0
                and row["se_boundary_absmax"] == 0.0,
                **row,
            )
            if device.type == "cuda":
                torch.cuda.empty_cache()


/var/folders/nm/2n89zz0x53gbk9546w6x328m0000gn/T/ipykernel_8136/3950201088.py:34: RuntimeWarning: model_gradient_sampling_interval > 1 uses a weighted temporal-sampling approximation; use interval 1 with float32 storage for exact gradients.
  result = DeepGPR.compute(


[PASS] long-run 2D order 2 on cpu
{
  "device": "cpu",
  "dimension": 2,
  "elapsed_seconds": 0.2307070420065429,
  "er_boundary_absmax": 0.0,
  "er_gradient_absmax": 524.3209228515625,
  "nt": 2000,
  "order": 2,
  "receiver_absmax": 780.2178955078125,
  "se_boundary_absmax": 0.0,
  "se_gradient_absmax": 9979.48828125
}
[PASS] long-run 2D order 4 on cpu
{
  "device": "cpu",
  "dimension": 2,
  "elapsed_seconds": 0.22824825000134297,
  "er_boundary_absmax": 0.0,
  "er_gradient_absmax": 405.99322509765625,
  "nt": 2000,
  "order": 4,
  "receiver_absmax": 753.9038696289062,
  "se_boundary_absmax": 0.0,
  "se_gradient_absmax": 8871.916015625
}
[PASS] long-run 2D order 8 on cpu
{
  "device": "cpu",
  "dimension": 2,
  "elapsed_seconds": 0.2483415419992525,
  "er_boundary_absmax": 0.0,
  "er_gradient_absmax": 378.2832946777344,
  "nt": 2000,
  "order": 8,
  "receiver_absmax": 751.2413330078125,
  "se_boundary_absmax": 0.0,
  "se_gradient_absmax": 8792.78515625
}
[PASS] long-run 3D order 2 o

In [ ]:
vu.save_report(
    "07_long_run_stability",
    CHECKS,
    METADATA,
    extra={"device_metadata": DEVICE_METADATA, "stress_rows": stress_rows},
)
print(f"Completed {len(CHECKS)} required checks.")


Report written to /Users/llsra/Desktop/DeepGPR/tests/results/07_long_run_stability.json
Completed 6 required checks.
